# MLP Medium Ensemble + Ridge

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

TEST_PATH = Path("data/test_features-1.csv")
PREPROCESSOR_PATH = Path("models/preprocessor_v2.joblib")
RIDGE_PATH = Path("models/best_baseline_v2.joblib")
ENSEMBLE_DIR = Path("models/medium_ensemble")
EXPECTED_PATH = Path("expected_output.csv")

Device: cpu


## Cargar validacion y modelos auxiliares

In [8]:
X_val = np.load("data/X_val_processed.npy")
y_val_usd = np.load("data/y_val_usd.npy").reshape(-1)

target_scaler = joblib.load("models/target_scaler_v2.joblib")
Y_MEAN = float(target_scaler["mean"])
Y_STD = float(target_scaler["std"])

ridge = joblib.load(RIDGE_PATH)
config = joblib.load(ENSEMBLE_DIR / "config.joblib")
SEEDS = config["seeds"]
HIDDEN = config["hidden_layers"]

print("Seeds:", SEEDS)
print("Arquitectura:", HIDDEN)

Seeds: [42, 123, 456, 789, 2026]
Arquitectura: [128, 64]


In [9]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout=0.1):
        super().__init__()
        layers=[]
        prev=input_dim
        for h in hidden_layers:
            layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]
            prev=h
        layers.append(nn.Linear(prev,1))
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x)

def load_medium(path, input_dim):
    ckpt=torch.load(path,map_location=DEVICE,weights_only=False)
    hp=ckpt["hyperparams"]
    model=MLP(input_dim, ckpt["hidden_layers"], dropout=hp["dropout"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model

def predict_usd(model, X):
    xt=torch.tensor(X,dtype=torch.float32,device=DEVICE)
    with torch.no_grad():
        pred_scaled=model(xt).squeeze(1).cpu().numpy()
    return pred_scaled * Y_STD + Y_MEAN

## Predicciones en validacion

In [10]:
val_preds=[]
for seed in SEEDS:
    path=ENSEMBLE_DIR / f"medium_seed_{seed}.pt"
    model=load_medium(path, X_val.shape[1])
    val_preds.append(predict_usd(model,X_val))

mlp_val=np.mean(val_preds,axis=0)
ridge_val=ridge.predict(X_val).reshape(-1)

def rmse(y,p):
    return float(np.sqrt(mean_squared_error(y,p)))

print(f"RMSE Ridge validacion:        ${rmse(y_val_usd,ridge_val):,.2f}")
print(f"RMSE MLP Ensemble validacion: ${rmse(y_val_usd,mlp_val):,.2f}")

RMSE Ridge validacion:        $31,360.03
RMSE MLP Ensemble validacion: $23,133.46


## Elegir peso del blend 

In [11]:
rows=[]
for w_mlp in np.arange(0.0,1.0001,0.025):
    pred=w_mlp*mlp_val + (1-w_mlp)*ridge_val
    rows.append({"peso_mlp":w_mlp,"peso_ridge":1-w_mlp,"RMSE_USD":rmse(y_val_usd,pred)})

blend_df=pd.DataFrame(rows).sort_values("RMSE_USD").reset_index(drop=True)
display(blend_df.head(10))

best_w=float(blend_df.iloc[0]["peso_mlp"])
best_rmse=float(blend_df.iloc[0]["RMSE_USD"])
print(f"Mejor peso MLP:   {best_w:.3f}")
print(f"Mejor peso Ridge: {1-best_w:.3f}")
print(f"RMSE Blend Val:   ${best_rmse:,.2f}")

,peso_mlp,peso_ridge,RMSE_USD
0,1.000,0.000,23133.460682
1,0.975,0.025,23157.510102
2,0.950,0.050,23192.696927
3,0.925,0.075,23238.970567
4,0.900,0.100,23296.264956
5,0.875,0.125,23364.499019
6,0.850,0.150,23443.577234
7,0.825,0.175,23533.390284
8,0.800,0.200,23633.815787
9,0.775,0.225,23744.719089


Mejor peso MLP:   1.000
Mejor peso Ridge: 0.000
RMSE Blend Val:   $23,133.46


## Cargar test y preprocesar

In [12]:
test_df=pd.read_csv(TEST_PATH)
assert "Id" in test_df.columns, "El test debe contener la columna Id"
ids=test_df["Id"].copy()
X_raw=test_df.drop(columns=["Id", "SalePrice"], errors="ignore")
preprocessor=joblib.load(PREPROCESSOR_PATH)
X_test=preprocessor.transform(X_raw)
if hasattr(X_test,"toarray"):
    X_test=X_test.toarray()
X_test=np.asarray(X_test,dtype=np.float32)
print("Test procesado:",X_test.shape)

Test procesado: (292, 278)


## Predecir test 

In [13]:
test_preds=[]
for seed in SEEDS:
    path=ENSEMBLE_DIR / f"medium_seed_{seed}.pt"
    model=load_medium(path, X_test.shape[1])
    test_preds.append(predict_usd(model,X_test))

mlp_test=np.mean(test_preds,axis=0)
ridge_test=ridge.predict(X_test).reshape(-1)
blend_test=best_w*mlp_test + (1-best_w)*ridge_test

predictions=pd.DataFrame({"Id":ids.to_numpy(),"Prediction":blend_test})
assert predictions.columns.tolist()==["Id","Prediction"]
assert len(predictions)==len(test_df)
assert predictions["Prediction"].notna().all()

if EXPECTED_PATH.exists():
    expected=pd.read_csv(EXPECTED_PATH)
    assert predictions.columns.tolist()==expected.columns.tolist(), f"Columnas esperadas: {expected.columns.tolist()}"
    assert len(predictions)==len(expected), "La cantidad de filas no coincide con expected_output.csv"
    assert np.array_equal(predictions["Id"].to_numpy(),expected["Id"].to_numpy()), "Los Id no coinciden con expected_output.csv"
    print("Formato verificado contra expected_output.csv")

predictions.to_csv("predictions.csv",index=False)
print("Archivo listo: predictions.csv")
print("Columnas:",predictions.columns.tolist())
print("Filas:",len(predictions))
display(predictions.head())

Archivo listo: predictions.csv
Columnas: ['Id', 'Prediction']
Filas: 292


,Id,Prediction
0,893,154943.890625
1,1106,341251.593750
2,414,99198.085938
3,523,172828.921875
4,1037,352010.312500
